# Pipeline

In [ ]:
%load_ext autoreload
%autoreload 2

import os.path as osp, sys

PATH = osp.abspath('../src')
if PATH not in sys.path:
    sys.path.append(PATH)

from nlp_dashboard import DataPipeline
from nlp_dashboard.cli.pipeline import load_dates, write_artifacts_to_disk

## Configuration

### Set hyperparameters and initialize

Examples of `hf` and `spacy` models provided below. For details, see official documentation: [Hugging Face](https://huggingface.co/models) and [spaCy](https://spacy.io/usage/models).

In [ ]:
hyperparams = dict(
    hf__model="distilbert-base-cased",
    spacy__model="en_core_web_sm",
    processor__dates="",
    # hf__chunksize=None,
    # hf__device="auto",
    # hf__max_workers=None,
    # kmeans__device="auto",
    # kmeans__distance=None,
    # kmeans__elbow_tol=0.2,
    # kmeans__init="k-means++",
    # kmeans__max_iter=300,
    # kmeans__minibatch=False,
    # kmeans__n_clusters=(2,8),
    # kmeans__n_init=20,
    # kmeans__preserve_embeddings=True,
    # kmeans__seed=None,
    # kmeans__tol=1e-4,
    # kmeans__verbose=False,
    # nltk__min_gram_size=2,
    # pandas__applymap=lambda x: x,
    # pandas__column="text",
    # pandas__compression="infer",
    # pandas__drop_duplicates=False,
    # pandas__dropna=False,
    # pandas__engine=None,
    # pandas__json_lines=False,
    # pandas__json_normalize=False,
    # pandas__low_memory=True,
    # pandas__nrows=None,
    # pandas__sep=None,
    # pandas__sort=[],
    # processor__custom_stopwords=None,
    # processor__json=False,
    # processor__max_groups=100,
    # processor__max_objects=100,
    # processor__min_time_steps=3,
    # processor__stopwords=True,
    # spacy__batch_size=500,
    # spacy__device="auto",
    # spacy__min_lemma_size=2,
    # spacy__n_process=1,
    # tokenizer__clear_accents=False,
    # tokenizer__clear_characters=False,
    # tokenizer__clear_emojis=False,
    # tokenizer__ignore_numbers=True,
    # tsne__n_components=3,
    # tsne__perplexity=60.0,
    # umap__metric="euclidean",
    # umap__min_dist=0.1,
    # umap__n_components=2,
    # umap__n_neighbors=15,
)

pipeline = DataPipeline(
    use_hf=True,
    use_kmeans=True,
    use_nltk=True,
    use_pandas=True,
    use_processor=True,
    use_reader=True,
    use_spacy=True,
    use_tokenizer=True,
    # use_tsne=True,
    use_umap=True,
    named_steps=[],
    seed=None,
)

pipeline.set_passthroughs(**hyperparams)

### Path of file to analyze

In [ ]:
X = ""

### Process data

In [ ]:
artifacts = pipeline.fit_transform(X, **hyperparams|{"processor__dates": load_dates(X, hyperparams["processor__dates"]) if "processor" in pipeline.named_steps and hyperparams.get("processor__dates") else None})
list(artifacts)

### Save artifacts

In [ ]:
write_artifacts_to_disk(artifacts)

### View artifacts

In [ ]:
# artifacts["name_of_artifact"]

___

## Test

### Step by step

In [ ]:
from time import time

def test(X, named_steps=[]):
    X_ = []
    t0 = time()
    pipeline = DataPipeline(named_steps=["reader", "pandas", "tokenizer", "nltk", "spacy", "hf", "umap", "kmeans", "processor"])  # "tsne"

    for i, step in enumerate(named_steps or pipeline.named_steps):
        t = time()
        print(f"\nstep {i+1}: '{step}'\ninput: {type(X_[-1] if X_ else X)}")

        # %time \
        X_.append(
            pipeline.named_steps[step]
            .fit(X_[-1] if X_ else X, **{k.split("_")[-1]: v for k, v in fit_params.items() if k.startswith(step)})
            .transform(X_[-1] if X_ else X)
        )

        print(f"output: {type(X_[-1] if X_ else X)}\ntime: {float(time()-t):.3f}s")

    print(f"\nFinished execution in {float(time()-t0):.3f}s.")
    return {step: X_[i] for i, step in enumerate(named_steps or pipeline.named_steps)}

X_ = test(X)  # named_steps=["reader", "pandas", "tokenizer", "nltk", "spacy", "hf", "processor"]

list(X_)